# Documentation: Patient Heart Rate Simulator (v1.0)

## 1. Overview
This notebook serves as a **Stochastic Data Generator** for a Healthcare IoT pipeline. It simulates high-fidelity heart rate telemetry for a single patient, generating exactly **200 data points**, each point being 1 min, as individual JSON files. The generator is designed to test downstream Spark logic, including Auto Loader ingestion, windowed aggregations, and real-time alerting.

---

## 2. Core Logic & Methodology
The generator uses **Object-Oriented Programming (OOP)** to maintain a persistent state, ensuring that each data point is physically "aware" of the previous reading.

### Physiological Modeling
* **State-Switching:** The patient transitions between three physiological states: `REST`, `EXERCISE`, and `CRISIS`. 
* **Forced Event:** To ensure meaningful test data, an exercise event is **hard-coded** between Sequence IDs 70 and 130.
* **Stochastic Autoregression (ARIMA-style):** Heart rate is calculated using the formula:
  $$BPM_{t} = BPM_{t-1} + \phi(Target - BPM_{t-1}) + \sigma$$
  * **Drift ($\phi$):** A 15% pull toward the current state's target BPM.
  * **Volatility ($\sigma$):** Gaussian noise (Normal Distribution) to simulate natural Heart Rate Variability (HRV).

---

## 3. Data Schema
The output is saved in JSON format with the following structure:

| Field | Type | Description |
| :--- | :--- | :--- |
| `patient_id` | String | Unique identifier (e.g., "PT-001") |
| `timestamp` | ISO-8601 | The wall-clock time of generation |
| `heart_rate_bpm` | Float | The simulated heart rate value |
| `current_state` | String | Physiological state (`REST`, `EXERCISE`, `CRISIS`) |
| `sequence_id` | Integer | Incremental counter (1–200) for strict ordering |

---

## 4. Infrastructure & Storage
* **Catalog:** `production_catalog`
* **Schema:** `patient_data`
* **Storage Type:** Unity Catalog Volume (`raw_jsons`)
* **Path:** `/Volumes/production_catalog/patient_data/raw_jsons/`
* **File Naming:** `hr_batch_{sequence_id:03d}.json` (e.g., `hr_batch_045.json`)

---

## 5. Usage Instructions
1. Ensure the **Unity Catalog Volume** exists before running.
2. Run the Python generator cell to populate the volume.
3. Each json file correspond to a state of a sequence simulated heart rate for a "patient".

# Create a Schema for the data generator

In [0]:
%sql
-- This creates the catalog only if it doesn't already exist
CREATE CATALOG IF NOT EXISTS production_catalog;

-- Optional: Add a comment for metadata clarity
COMMENT ON CATALOG production_catalog IS 'The "Patient" data catalog';

# Data generator data

## Create a Volume Path

In [0]:
import json
import time
import random
import os
from datetime import datetime

# 1. Variables
catalog_name = "production_catalog"
schema_name = "patient_data"
volume_name = "raw_jsons"

# 2. Create the Catalog if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

# 3. Create the Schema (Database) inside that catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# 4. Create the Volume inside the schema
# We use 'CREATE VOLUME' for a Managed Volume (easiest setup)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

# 5. Now you can use dbutils to create sub-folders inside that volume
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/"
dbutils.fs.mkdirs(volume_path)

print(f"Success! Path is ready: {volume_path}")

## Main Code

In [0]:
import json
import time
import random
import os
from datetime import datetime

# --- SETTINGS ---
CATALOG = "production_catalog"
SCHEMA = "patient_data"
VOLUME = "raw_jsons"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"

# --- CLEANUP FOR DATABRICKS ---
try:
    files = dbutils.fs.ls(volume_path)
    for file in files:
        dbutils.fs.rm(file.path)
    print(f"Cleanup Complete: Removed {len(files)} files.")
except Exception as e:
    print("Volume is already empty or does not exist.")

# Ensure volume exists
os.makedirs(VOLUME_PATH, exist_ok=True)

class ContinuousHeartMonitor:
    def __init__(self, patient_id):
        self.patient_id = patient_id
        self.bpm = 72.0       # Initial BPM
        self.state = "REST"    # Start at rest
        self.tick_count = 0

    def update_bpm(self):
        """
        Simulates the logic of a State-Switching Model (like a Random Forest 
        deciding state) + ARIMA (for the smooth transition of values).
        """
        # 1. Randomly transition states to simulate a 'Day in the Life'
        roll = random.random()
        if roll < 0.005: self.state = "CRISIS"    # Rare event
        elif roll < 0.03: self.state = "EXERCISE" # Occasional activity
        elif roll < 0.10: self.state = "REST"     # Return to baseline

        # 2. Define target zones for each state
        targets = {"REST": 70, "EXERCISE": 150, "CRISIS": 195}
        volatility = {"REST": 1.2, "EXERCISE": 4.5, "CRISIS": 9.0}
        
        target = targets[self.state]
        sigma = volatility[self.state]

        # 3. ARIMA-style Autoregression: BPM(t) = BPM(t-1) + Drift + Noise
        drift = (target - self.bpm) * 0.1  # Slowly pull toward the state's target
        noise = random.normalvariate(0, sigma)
        
        self.bpm += drift + noise
        
        # Physical bounds (Safety check)
        self.bpm = max(40, min(220, self.bpm))
        self.tick_count += 1
        
        return {
            "patient_id": self.patient_id,
            "timestamp": datetime.now().isoformat(),
            "heart_rate_bpm": round(self.bpm, 2),
            "current_state": self.state,
            "sequence_id": self.tick_count
        }

# --- EXECUTION ---
patient_sim = ContinuousHeartMonitor("PT-001")
total_points = 100

print(f"Generating {total_points} points for patient {patient_sim.patient_id}...")

try:
    # Changed from 'while True' to a fixed range
    for i in range(1, total_points + 1):
        data = patient_sim.update_bpm()
        
        # Save unique file for each record
        # Padding the filename with zeros (e.g., 001, 002) helps with sorting
        file_name = f"hr_batch_{i:03d}.json"
        full_path = os.path.join(VOLUME_PATH, file_name)
        
        with open(full_path, "w") as f:
            json.dump(data, f)
            
        if i % 20 == 0:
            print(f"Generated {i}/{total_points} | {data['heart_rate_bpm']} BPM | State: {data['current_state']}")
            
        # Small delay so it finishes in a few seconds
        time.sleep(5) 

except Exception as e:
    print(f"An error occurred: {e}")

print(f"\n--- Done! 200 points saved to {VOLUME_PATH} ---")